## CELL 1: SETUP & CONFIGURATION

In [0]:
import logging
from pyspark.sql import SparkSession

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def get_param(param_name: str, default_value: str) -> str:
    try:
        return dbutils.widgets.get(param_name)
    except Exception:
        return default_value

# Pull from orchestrator (variables.yml) or fallback to local defaults
CATALOG = get_param("catalog_name", "main")
SCHEMA = get_param("schema_name", "fixkart_prd")
RBAC_PRINCIPAL = get_param("rbac_principal", "account users")

# Dynamic Table Paths
FACT_TABLE = f"{CATALOG}.{SCHEMA}.fact_work_orders"
DIM_TECH_TABLE = f"{CATALOG}.{SCHEMA}.dim_technician"
DIM_CUST_TABLE = f"{CATALOG}.{SCHEMA}.dim_customer"
KPI_VIEW = f"{CATALOG}.{SCHEMA}.semantic_executive_kpis"

logger.info(f"Initialized Governance Pipeline | Target Schema: {CATALOG}.{SCHEMA}")

## CELL 2: CONSTRAINTS & LINEAGE

In [0]:
def safe_execute_ddl(spark: SparkSession, query: str) -> None:
    """Executes DDL securely, ignoring 'already exists' states for idempotency."""
    try:
        spark.sql(query)
    except Exception as e:
        error_msg = str(e).lower()
        if "already exists" not in error_msg and "already has a mask" not in error_msg:
            raise

def apply_governance_rules(spark: SparkSession) -> None:
    logger.info("Applying Unity Catalog lineage tags and constraints...")
    
    # 1. Tags & Lineage Enrichment
    safe_execute_ddl(spark, f"ALTER TABLE {FACT_TABLE} SET TBLPROPERTIES ('business_unit' = 'operations', 'contains_pii' = 'true')")
    safe_execute_ddl(spark, f"COMMENT ON TABLE {FACT_TABLE} IS 'Central fact table for all field dispatches.'")
    safe_execute_ddl(spark, f"COMMENT ON COLUMN {FACT_TABLE}.customer_id IS 'Unique identifier for the customer (PII).'")

    # 2. NOT NULL & Check Constraints
    safe_execute_ddl(spark, f"ALTER TABLE {FACT_TABLE} ALTER COLUMN work_order_id SET NOT NULL")
    safe_execute_ddl(spark, f"ALTER TABLE {FACT_TABLE} ADD CONSTRAINT valid_status CHECK (status IN ('COMPLETED', 'FAILED', 'CANCELLED'))")

    # 3. Primary & Foreign Keys
    safe_execute_ddl(spark, f"ALTER TABLE {DIM_TECH_TABLE} ADD CONSTRAINT dim_tech_pk PRIMARY KEY(technician_id)")
    safe_execute_ddl(spark, f"ALTER TABLE {DIM_CUST_TABLE} ADD CONSTRAINT dim_cust_pk PRIMARY KEY(customer_id)")
    safe_execute_ddl(spark, f"ALTER TABLE {FACT_TABLE} ADD CONSTRAINT fact_wo_pk PRIMARY KEY(work_order_id)")
    safe_execute_ddl(spark, f"ALTER TABLE {FACT_TABLE} ADD CONSTRAINT fk_fact_tech FOREIGN KEY(technician_id) REFERENCES {DIM_TECH_TABLE}")
    
    logger.info("Governance rules applied successfully.")

## CELL 3: DYNAMIC DATA MASKING (DDM) & CLS

In [0]:
def apply_security_policies(spark: SparkSession, principal: str) -> None:
    logger.info(f"Configuring DDM and applying RBAC to principal: `{principal}`...")
    
    # 1. Dynamic Data Masking (DDM)
    masking_func_sql = f"""
    CREATE FUNCTION IF NOT EXISTS {CATALOG}.{SCHEMA}.mask_pii(val STRING)
    RETURN CASE WHEN is_account_group_member('admin') OR is_account_group_member('executives') 
                THEN val 
                ELSE '***-REDACTED-***' 
           END;
    """
    spark.sql(masking_func_sql)
    safe_execute_ddl(spark, f"ALTER TABLE {DIM_CUST_TABLE} ALTER COLUMN customer_id SET MASK {CATALOG}.{SCHEMA}.mask_pii")

    # Role-Based Access Control (RBAC)
    try:
        spark.sql(f"GRANT SELECT ON VIEW {KPI_VIEW} TO `{principal}`")
        spark.sql(f"GRANT SELECT ON TABLE {FACT_TABLE} TO `{principal}`")
        spark.sql(f"GRANT SELECT ON TABLE {DIM_TECH_TABLE} TO `{principal}`")
        logger.info("Security policies and grants successfully applied.")
    except Exception as e:
        logger.warning(f"Could not grant permissions to `{principal}`. Error: {str(e)}")

## CELL 4: ORCHESTRATION & EXECUTION

In [0]:
if 'spark' in locals() or 'spark' in globals():
    try:
        logger.info("--- Starting Governance & Security Pipeline ---")
        
        apply_governance_rules(spark)
        apply_security_policies(spark, RBAC_PRINCIPAL)
        
        logger.info("--- Governance Pipeline Completed Successfully ---")
        
    except Exception as main_e:
        logger.error(f"Governance Pipeline execution aborted: {str(main_e)}")
else:
    logger.warning("SparkSession not found.")